In [8]:
import pandas as pd

# Caminhos dos ficheiros processados em 1.8.2 (Scaled) e 1.8.1 (Unscaled/Raw Dates)
trainFile = "../../data/preprocessing/1_8_2.csv"
testFile = "../../data/preprocessing/1_8_2_test.csv"

trainFileUnscaled = "../../data/preprocessing/1_8_1.csv"
testFileUnscaled = "../../data/preprocessing/1_8_1_test.csv"

# Carregar os datasets escalados para as comparações estruturais
train = pd.read_csv(trainFile)
test = pd.read_csv(testFile)

# Excluir identificadores e ruído da comparação de duplicados
columns = [c for c in test.columns if c not in ["ID", "random_noise"]]

# 1. Repeated rows inside each file
print(f"repeated rows inside train: {train.duplicated(subset=columns).sum()}")
print(f"repeated rows inside test:  {test.duplicated(subset=columns).sum()}")

# 2. Test rows that are a copy of a train row
copies = test.merge(train[columns].drop_duplicates(), on=columns, how="inner")
print(f"test rows identical to a train row: {len(copies)}")

# 3. Check for same ID in both files
print(f"IDs in both files: {test['ID'].isin(train['ID']).sum()}")

# 4. Customers in both files
if "customer_id" in test.columns and "customer_id" in train.columns:
  in_train = test["customer_id"].isin(train["customer_id"])
  print(
      f"test customers also in train:"
      f" {test.loc[in_train, 'customer_id'].nunique()} of"
      f" {test['customer_id'].nunique()} ({in_train.sum()} of {len(test)} test"
      " rows)"
  )

# 5. Dates: Carregar atributos temporais não escalados para reconstrução correta
date_cols = ["purchase_year", "purchase_month", "purchase_day"]

try:
  # Tenta ler do ficheiro 1.8.1 (onde os inteiros da data estão intactos)
  train_raw = pd.read_csv(trainFileUnscaled, usecols=date_cols)
  test_raw = pd.read_csv(testFileUnscaled, usecols=date_cols)

  train_dates = pd.to_datetime(
      train_raw.rename(
          columns={
              "purchase_year": "year",
              "purchase_month": "month",
              "purchase_day": "day",
          }
      )
  )
  test_dates = pd.to_datetime(
      test_raw.rename(
          columns={
              "purchase_year": "year",
              "purchase_month": "month",
              "purchase_day": "day",
          }
      )
  )

  print(
      f"train period: {train_dates.min().date()} to {train_dates.max().date()}"
  )
  print(
      f"test period:  {test_dates.min().date()} to {test_dates.max().date()}"
  )

except Exception as e:
  print(
      "\nNota: Não foi possível calcular o intervalo exacto de datas através do"
      " ficheiro escalado."
  )
  print(f"Mensagem de salvaguarda: {e}")


repeated rows inside train: 0
repeated rows inside test:  0
test rows identical to a train row: 0
IDs in both files: 0
test customers also in train: 184 of 206 (932 of 986 test rows)
train period: 2015-07-11 to 2025-02-15
test period:  2025-05-24 to 2026-10-06
